# 生产运行时：队列、事件、时程

生产级agents有六种运行时形态：request-response, streaming, durable-execution, queue-based background, event-driven 以及 schedualed。在你选框架之前选择运行时形态。观测性在每种形态中多是承重的。

## 问题描述

生产级agents面临的问题是Jupyter Notebook 没有面临的：在第37步网络超时，用户语音输入途中挂掉了，时程任务因为系统重启挂掉了，后台任务跑光了内存。运行时形态决定哪些失败能存活。

## 基本概念

|模式|概要|适合场景|观察哪些东西|
|---|---|---|---|
|Request-Response|同步的HTTP请求，用户等待补全|短任务|HTTP访问日志+OTel spans|
|Streaming|SSE 或者WebSocket用于渐进式输出|-|每个chunk计时，头延迟、尾延迟|
|Durable execution|每步做状态检查点，失败自动恢复|步数未知，从零恢复开销大|-|
|Queue-based|工作进入队列，然后由workder拾起，结果流回|长程任务|队列深度、任务延迟分布、DLQ大小（多次运行仍然失败的任务）|
|Event-driven|agents订阅触发器（如来了新邮件...）|-|触发源、事件到开始延迟、agent延迟|
|Scheduled|通常与durable execution结合便于失败后下一个时间再跑|需要定时跑的时程agents|-|

### 什么时候生产运行时失效

- 选了错误的模式。
- 没有DLQ。 没办法跟踪失败的任务
- 后台工作不透明。  不对后台agent运行做追踪报道，错误只有在用户报告的时候才可见。
- 跳过持久状态。  大于30秒且你无法承受从头来过的任务需要durable execution。

# 开始编码

对应本章核心：**六种运行时形态选择**、**Queue-based：重试 + 退避 + DLQ**、**Durable execution：检查点 + resume**、**Streaming：逐 chunk 计时**、**Event/Scheduled 触发器**。  
先用内存玩具跑通队列/DLQ、检查点恢复、流式计时；再用 **LangGraph MemorySaver + DeepSeek** 做生产 durable execution（不硬凑 PyTorch）。无 `DEEPSEEK_API_KEY` 则生产示例 SKIP。


## 1. 教学玩具：队列 + DLQ + 检查点 + 流式

- **Queue**：worker 取任务，失败按退避重试；用尽后进 DLQ。
- **DLQ**：落死信后不再自动消费，可重放。
- **Durable**：每步 checkpoint；崩溃后从最后一个 checkpoint 续跑，不重复已完成的副作用。
- **Streaming**：记录首 chunk / 尾 chunk 延迟。


In [ ]:
from __future__ import annotations

import heapq
import json
import time
import uuid
from dataclasses import dataclass, field
from typing import Any, Callable, Literal

RuntimePattern = Literal[
    "request_response", "streaming", "durable",
    "queue_based", "event_driven", "scheduled",
]


def pick_runtime(
    *,
    user_waits: bool = False,
    long_running: bool = False,
    cannot_restart_from_zero: bool = False,
    needs_stream: bool = False,
    has_trigger: bool = False,
    periodic: bool = False,
) -> RuntimePattern:
    """
    先选运行时形态，再选框架。

    Returns:
        pattern: 推荐运行时。
    """
    if periodic:
        return "scheduled"
    if has_trigger:
        return "event_driven"
    if cannot_restart_from_zero:
        return "durable"
    if long_running and not user_waits:
        return "queue_based"
    if needs_stream:
        return "streaming"
    return "request_response"


# ---------- Queue-based + DLQ ----------

@dataclass
class Task:
    """队列任务。"""

    id: str
    payload: str
    attempts: int = 0
    not_before: float = 0.0


@dataclass
class DLQEntry:
    """死信：重试用尽仍失败的任务。"""

    task: Task
    error: str
    failed_at: float = 0.0


class QueueRuntime:
    """内存队列：重试 + 指数退避 + DLQ。"""

    def __init__(self, *, max_attempts: int = 3, base_delay: float = 0.1) -> None:
        self.max_attempts = max_attempts
        self.base_delay = base_delay
        self._heap: list[tuple[float, str, Task]] = []
        self.done: list[str] = []
        self.dlq: list[DLQEntry] = []
        self.sleep_total: float = 0.0  # 记录退避而不真睡

    def submit(self, payload: str) -> str:
        """
        Args:
            payload: 任务内容。

        Returns:
            task_id。
        """
        t = Task(id=f"t_{uuid.uuid4().hex[:8]}", payload=payload)
        heapq.heappush(self._heap, (0.0, t.id, t))
        return t.id

    def drain(self, worker: Callable[[str], str]) -> None:
        """
        消费到空。worker 抛异常 → 重试或入 DLQ。

        Args:
            worker: 处理函数。
        """
        while self._heap:
            _, _, task = heapq.heappop(self._heap)
            task.attempts += 1
            try:
                out = worker(task.payload)
                self.done.append(f"{task.id}={out}")
            except Exception as e:
                if task.attempts >= self.max_attempts:
                    self.dlq.append(
                        DLQEntry(task=task, error=str(e), failed_at=time.time())
                    )
                else:
                    delay = self.base_delay * (2 ** (task.attempts - 1))
                    self.sleep_total += delay
                    task.not_before = time.time() + delay
                    heapq.heappush(self._heap, (task.not_before, task.id, task))

    def replay_dlq(self, worker: Callable[[str], str] | None = None) -> int:
        """
        把 DLQ 里的任务重新入队（修复上游后重放）。

        Returns:
            n: 重放条数。
        """
        n = len(self.dlq)
        for entry in self.dlq:
            t = entry.task
            t.attempts = 0
            heapq.heappush(self._heap, (0.0, t.id, t))
        self.dlq.clear()
        return n


# ---------- Durable execution ----------

class DurableRuntime:
    """每步写检查点；崩溃后从下一未完成步骤续跑。"""

    def __init__(self) -> None:
        self.checkpoints: dict[str, dict[str, Any]] = {}

    def run(
        self,
        session_id: str,
        steps: list[Callable[[dict[str, Any]], None]],
        state: dict[str, Any] | None = None,
        *,
        fail_at: int | None = None,
    ) -> tuple[dict[str, Any], bool]:
        """
        Args:
            session_id: 会话。
            steps: 步骤函数，原地改 state。
            state: 可选已有状态（恢复时传入）。
            fail_at: 演示崩溃：在第几步抛异常。

        Returns:
            (state, completed)。
        """
        ckpt = self.checkpoints.get(session_id)
        if ckpt is not None:
            state = ckpt["state"]
            done = ckpt["done"]
        else:
            state = state if state is not None else {}
            state.setdefault("log", [])
            done = 0
        for i in range(done, len(steps)):
            if fail_at is not None and i == fail_at:
                raise RuntimeError(f"crash at step {i}")
            steps[i](state)
            state["log"].append(f"step{i}")
            done = i + 1
            # 每步后写检查点
            self.checkpoints[session_id] = {
                "state": json.loads(json.dumps(state)),
                "done": done,
            }
        return state, True


# ---------- Streaming ----------

@dataclass
class Chunk:
    """流式输出的一片。"""

    idx: int
    text: str
    t: float


@dataclass
class StreamReport:
    """流式计时。"""

    chunks: list[Chunk] = field(default_factory=list)

    @property
    def time_to_first_chunk(self) -> float | None:
        return self.chunks[0].t if self.chunks else None

    @property
    def total_latency(self) -> float | None:
        return self.chunks[-1].t if self.chunks else None


def stream_words(text: str, *, tick: float = 0.01) -> StreamReport:
    """
    玩具流式：按词切片，记录每片相对时间。

    Returns:
        report: 含首/尾延迟。
    """
    rep = StreamReport()
    start = time.monotonic()
    for i, w in enumerate(text.split()):
        time.sleep(tick)
        rep.chunks.append(Chunk(idx=i, text=w, t=time.monotonic() - start))
    return rep


# ---------- Event / Scheduled ----------

@dataclass
class Trigger:
    """触发源（新邮件、cron 到点等）。"""

    source: str
    fired_at: float


class EventBus:
    """极简事件总线：订阅 → 触发 → 记录入队延迟。"""

    def __init__(self) -> None:
        self.subs: dict[str, list[Callable[[Trigger], Any]]] = {}
        self.enqueued_at: dict[str, float] = {}

    def subscribe(self, source: str, fn: Callable[[Trigger], Any]) -> None:
        self.subs.setdefault(source, []).append(fn)

    def fire(self, source: str) -> list[Any]:
        trig = Trigger(source=source, fired_at=time.monotonic())
        self.enqueued_at[source] = time.monotonic()
        return [fn(trig) for fn in self.subs.get(source, [])]

    def event_to_start_latency(self, source: str, started_at: float) -> float:
        return started_at - self.enqueued_at.get(source, started_at)


print("runtime toys ready | queue+DLQ + durable + stream + events")


## 2. 玩具示例：退避、DLQ、崩溃恢复、流式计时


In [ ]:
def demo_runtime_toy() -> None:
    """断言队列退避/DLQ、检查点恢复不重放副作用、流式计时、事件延迟。"""
    # 运行时选择
    assert pick_runtime(user_waits=True) == "request_response"
    assert pick_runtime(needs_stream=True) == "streaming"
    assert pick_runtime(long_running=True) == "queue_based"
    assert pick_runtime(cannot_restart_from_zero=True) == "durable"
    assert pick_runtime(has_trigger=True) == "event_driven"
    assert pick_runtime(periodic=True) == "scheduled"
    print("pick_runtime ok")

    # 队列：前两次失败，第三次成功 → 不进 DLQ，累计退避
    q = QueueRuntime(max_attempts=3, base_delay=0.1)
    calls = {"n": 0}

    def flaky(p: str) -> str:
        calls["n"] += 1
        if calls["n"] < 3:
            raise RuntimeError("boom")
        return f"ok:{p}"

    q.submit("job")
    q.drain(flaky)
    assert len(q.done) == 1 and q.dlq == []
    assert calls["n"] == 3
    assert abs(q.sleep_total - (0.1 + 0.2)) < 1e-6
    print("retry+backoff ok (2 retries, delay 0.1+0.2)")

    # 永远失败 → DLQ，然后修复 worker 后重放
    q2 = QueueRuntime(max_attempts=2, base_delay=0.01)
    q2.submit("bad")
    q2.drain(lambda p: (_ for _ in ()).throw(RuntimeError("always")))
    assert len(q2.dlq) == 1
    n = q2.replay_dlq()
    assert n == 1 and q2.dlq == []
    q2.drain(lambda p: "fixed")
    assert q2.done and q2.dlq == []
    print("DLQ + replay ok")

    # Durable：第 2 步崩溃；resume 不重复已完成副作用
    dur = DurableRuntime()
    side_effects: list[str] = []

    def make_step(i: int):
        def step(state: dict[str, Any]) -> None:
            side_effects.append(f"effect{i}")  # 副作用只应发生一次
            state[f"k{i}"] = i
        return step

    steps = [make_step(i) for i in range(4)]
    try:
        dur.run("s1", steps, fail_at=2)
    except RuntimeError:
        pass
    assert side_effects == ["effect0", "effect1"]  # 崩溃前只做了 0,1
    state, done = dur.run("s1", steps)  # 恢复
    assert done and side_effects == ["effect0", "effect1", "effect2", "effect3"]
    assert state["log"] == ["step0", "step1", "step2", "step3"]
    print("durable resume without side-effect replay ok")

    # Streaming：首 chunk 远早于尾 chunk
    rep = stream_words("hello brave new agent world", tick=0.005)
    assert rep.time_to_first_chunk is not None and rep.total_latency is not None
    assert rep.time_to_first_chunk < rep.total_latency
    assert len(rep.chunks) == 5
    print(f"streaming ttfc={rep.time_to_first_chunk:.4f}s total={rep.total_latency:.4f}s")

    # Event：订阅被触发，延迟可测
    bus = EventBus()
    seen: list[str] = []
    bus.subscribe("email", lambda t: seen.append(t.source))
    bus.fire("email")
    assert seen == ["email"]
    started = time.monotonic()
    assert bus.event_to_start_latency("email", started) >= 0
    print("event trigger + latency ok")
    print("TOY DEMO OK")


demo_runtime_toy()


## 3. 生产级：LangGraph Durable Execution + DeepSeek

`MemorySaver` 检查点：同一线程内图崩了能 resume；线程 id 即会话 id。外包一层队列 + DLQ，覆盖「长程后台任务」。需 `DEEPSEEK_API_KEY`。


In [ ]:
import json
import os
import sys
from pathlib import Path
from typing import Any, Literal

from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from typing_extensions import TypedDict

sys.path.append(str(Path("../../00_Common").resolve()))
from user_tools import load_project_env  # noqa: E402

load_project_env()

MODEL = "deepseek:deepseek-v4-flash"


class RunState(TypedDict, total=False):
    """生产图状态：三步，中间可注入故障。"""

    task: str
    facts: str
    draft: str
    final: str
    steps_done: list[str]


def get_llm(*, temperature: float = 0.0) -> Any:
    """
    Returns:
        llm: DeepSeek chat model。
    """
    if not os.getenv("DEEPSEEK_API_KEY"):
        raise RuntimeError("DEEPSEEK_API_KEY missing; copy .env.example → .env")
    return init_chat_model(
        MODEL,
        temperature=temperature,
        extra_body={"thinking": {"type": "disabled"}},
    )


STEP_CALLS: dict[str, int] = {}
INJECT_FAIL_AT: str | None = None  # 演示用：在哪个节点名注入故障


def research_node(state: RunState) -> dict[str, Any]:
    STEP_CALLS["research"] = STEP_CALLS.get("research", 0) + 1
    if INJECT_FAIL_AT == "research":
        raise RuntimeError("injected crash at research")
    out = str(
        get_llm().invoke(
            "List 2 short factual bullets in Chinese about:\n" + state["task"]
        ).content
    ).strip()
    return {
        "facts": out,
        "steps_done": list(state.get("steps_done") or []) + ["research"],
    }


def draft_node(state: RunState) -> dict[str, Any]:
    STEP_CALLS["draft"] = STEP_CALLS.get("draft", 0) + 1
    if INJECT_FAIL_AT == "draft":
        raise RuntimeError("injected crash at draft")
    out = str(
        get_llm().invoke(
            "Write <=40 Chinese characters using these facts.\n"
            f"Facts:\n{state['facts']}"
        ).content
    ).strip()
    return {
        "draft": out,
        "steps_done": list(state.get("steps_done") or []) + ["draft"],
    }


def polish_node(state: RunState) -> dict[str, Any]:
    STEP_CALLS["polish"] = STEP_CALLS.get("polish", 0) + 1
    if INJECT_FAIL_AT == "polish":
        raise RuntimeError("injected crash at polish")
    out = str(
        get_llm().invoke(
            "Polish to one sentence, <=30 Chinese characters.\n"
            f"Draft:\n{state['draft']}"
        ).content
    ).strip()
    return {
        "final": out,
        "steps_done": list(state.get("steps_done") or []) + ["polish"],
    }


CHECKPOINTER = MemorySaver()


def build_durable_graph() -> Any:
    """
    Returns:
        graph: 带检查点的三步 durable 图。
    """
    g = StateGraph(RunState)
    g.add_node("research", research_node)
    g.add_node("draft", draft_node)
    g.add_node("polish", polish_node)
    g.add_edge(START, "research")
    g.add_edge("research", "draft")
    g.add_edge("draft", "polish")
    g.add_edge("polish", END)
    return g.compile(checkpointer=CHECKPOINTER)


DURABLE_GRAPH = build_durable_graph()


class AgentJobQueue:
    """生产后台队列：跑 durable 图，失败重试，用尽入 DLQ。"""

    def __init__(self, *, max_attempts: int = 2) -> None:
        self.q = QueueRuntime(max_attempts=max_attempts, base_delay=0.05)
        self.results: dict[str, dict[str, Any]] = {}

    def submit_task(self, task: str, *, thread_id: str) -> str:
        """
        Args:
            task: 用户任务。
            thread_id: 检查点会话 id。

        Returns:
            job_id。
        """
        payload = json.dumps({"task": task, "thread_id": thread_id})
        return self.q.submit(payload)

    def drain(self) -> None:
        """worker：恢复 durable 图执行。"""

        def worker(payload: str) -> str:
            body = json.loads(payload)
            cfg = {"configurable": {"thread_id": body["thread_id"]}}
            snap = DURABLE_GRAPH.get_state(cfg)
            if snap.next:
                out = DURABLE_GRAPH.invoke(None, cfg)
            else:
                out = DURABLE_GRAPH.invoke(
                    {"task": body["task"], "steps_done": []}, cfg
                )
            self.results[body["thread_id"]] = out
            return out.get("final") or "no-final"

        self.q.drain(worker)

    def status(self) -> dict[str, Any]:
        return {
            "done": len(self.q.done),
            "dlq": len(self.q.dlq),
            "retry_delay_budget": round(self.q.sleep_total, 3),
        }


def run_durable_with_crash_demo(task: str, *, crash_at: str = "draft") -> dict[str, Any]:
    """
    一次任务：在指定节点注入崩溃，然后 resume（同一 thread_id），
    断言已完成节点不重跑。

    Returns:
        report: 含调用计数与最终结果。
    """
    global INJECT_FAIL_AT
    STEP_CALLS.clear()
    thread = f"demo-{uuid.uuid4().hex[:6]}"
    cfg = {"configurable": {"thread_id": thread}}
    INJECT_FAIL_AT = crash_at
    try:
        DURABLE_GRAPH.invoke({"task": task, "steps_done": []}, cfg)
    except RuntimeError:
        pass
    calls_after_crash = dict(STEP_CALLS)
    INJECT_FAIL_AT = None
    out = DURABLE_GRAPH.invoke(None, cfg)
    return {
        "thread_id": thread,
        "calls_after_crash": calls_after_crash,
        "calls_total": dict(STEP_CALLS),
        "steps_done": out.get("steps_done"),
        "final": out.get("final"),
    }


print(f"LangGraph durable runtime ready | {MODEL}")


## 4. 生产示例：崩溃 → resume，已完成节点不重跑

无 `DEEPSEEK_API_KEY` 则 SKIP。


In [ ]:
def demo_production_runtime() -> None:
    """生产：durable resume + 后台队列状态。"""
    if not os.getenv("DEEPSEEK_API_KEY"):
        print("SKIP production: DEEPSEEK_API_KEY missing")
        return

    rep = run_durable_with_crash_demo(
        "LangGraph durable execution 适合什么场景", crash_at="draft"
    )
    print("=== crash → resume ===")
    print(json.dumps(rep, ensure_ascii=False, indent=2)[:1200])
    # research 崩溃前已完成一次，resume 后不应再跑
    assert rep["calls_after_crash"].get("research") == 1
    assert rep["calls_total"].get("research") == 1
    assert rep["steps_done"] == ["research", "draft", "polish"]
    assert rep["final"]
    print("resume skipped completed node ok")

    # 后台队列包一层：正常任务跑通，状态可查
    jq = AgentJobQueue(max_attempts=2)
    tid = f"job-{uuid.uuid4().hex[:6]}"
    jq.submit_task("一句话说清死信队列的作用", thread_id=tid)
    jq.drain()
    st = jq.status()
    print("=== queue status ===", st)
    assert st["done"] == 1 and st["dlq"] == 0
    assert tid in jq.results
    print("PROD DEMO OK")


demo_production_runtime()
